### Co-purchasing analysis
Here we use a financial marketing dataset:

> Amazon product co-purchasing network and ground-truth communities
> 
> J. Yang and J. Leskovec. Defining and Evaluating Network Communities based on Ground-truth. ICDM, 2012.

To evaluate how the cluster of items (communities) can grow through the co-purchasing behavior of customers. 


In [1]:
import numpy as np
import scipy.sparse as sp
import pickle
from collections import defaultdict

max_id = 20000  # 设置最大node_ID

In [4]:
# 4. 生成标签
def generate_labels(communities):
    y = np.zeros((len(communities), max_id))
    for community_index, community in enumerate(communities):
        for node in community:
            y[community_index, node] = 1  # one-hot encoding
    return y

In [29]:
# 5. 保存数据
def save_data(graph, features, labels, test_indices, train_size):
    data = {
        'x': features[:train_size],  # 训练实例特征
        'tx': features[train_size:],  # 测试实例特征
        'allx': features,  # 所有特征
        'y': labels[:train_size],  # 训练标签
        'ty': labels[train_size:],  # 测试标签
        'ally': labels,  # 所有标签
        'graph': graph,  # 邻接表
        'test.index': test_indices  # 测试实例索引
    }
    
    with open('data/amazon/ind.amazon.x', 'wb') as f:
        pickle.dump(data['x'], f)

    with open('data/amazon/ind.amazon.tx', 'wb') as f:
        pickle.dump(data['tx'], f)

    with open('data/amazon/ind.amazon.allx', 'wb') as f:
        pickle.dump(data['allx'], f)

    with open('data/amazon/ind.amazon.y', 'wb') as f:
        pickle.dump(data['y'], f)

    with open('data/amazon/ind.amazon.ty', 'wb') as f:
        pickle.dump(data['ty'], f)

    with open('data/amazon/ind.amazon.ally', 'wb') as f:
        pickle.dump(data['ally'], f)

    with open('data/amazon/ind.amazon.graph', 'wb') as f:
        pickle.dump(data['graph'], f)

    with open('data/amazon/ind.amazon.test.index', 'wb') as f:
        pickle.dump(data['test.index'], f)
        #for index in data['test.index']:
            #f.write(f"{index}\n")

In [30]:
# 使用
graph = read_graph('data/graph.txt')
print("1")
communities = read_community('data/cmty.txt')
print("2")

features = build_feature_vectors(graph)
print("3")
labels = generate_labels(communities)
print("4")

train_size = 10000
test_indices = [i for i in range(train_size, max_id)]

save_data(graph, features, labels, test_indices, train_size)

1
2


Computing transition probabilities:   0%|          | 0/2329 [00:00<?, ?it/s]

3


MemoryError: Unable to allocate 2.72 GiB for an array with shape (18222, 20000) and data type float64

In [24]:
from cite_data_process import load_siteseer
import torch

%reload_ext autoreload
%autoreload 2

adj, features, labels, train_masks, val_masks, test_mask = load_siteseer("amazon")
print("Data set:\nAdjacency matrix is sparse: ", sp.issparse(adj))
print("All labels are one hot encoding: ", labels[0])
print("Each node has high dimentional feature: ", features.shape[1])
print("Total number of nodes: ", adj.shape[0])
print("Number of training nodes: ", train_masks[0].sum())
print("Number of validation nodes: ", val_masks[0].sum())
print("Number of testing nodes: ", test_mask.sum())

# convert labels from one-hot encoding to integer
class_num = labels.shape[1]
labels = np.argmax(labels, axis=1)

UnicodeDecodeError: 'gbk' codec can't decode byte 0x80 in position 0: illegal multibyte sequence

In [ ]:
import optuna
from sklearn.model_selection import KFold

# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    self_importance = trial.suggest_float('self_importance', 0.1, 2.0)
    hidden_units = trial.suggest_int('hidden_units', 8, 64)
    
    layer_dims = [features.shape[1], hidden_units, class_num]
    
    # KFold Cross Validation
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    val_acc_list = []
    
    for train_index, val_index in kf.split(features):
        # Generate masks for this fold
        train_mask = np.zeros(labels.shape[0], dtype=bool)
        val_mask = np.zeros(labels.shape[0], dtype=bool)
        train_mask[train_index] = True
        val_mask[val_index] = True
        
        # Train the model
        model, acc_list, loss_list = train_base(
            layer_dims=layer_dims,
            data=(adj, features, labels, train_mask, val_mask),
            dropout_rate=dropout_rate,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            self_importance=self_importance,
            epoch_num=100
        )
        
        # Use the validation accuracy for this fold
        val_acc = acc_list[-1] #acc_list的最后一个
        val_acc_list.append(val_acc)
    
    # Return the mean validation accuracy across all folds
    mean_val_acc = np.mean(val_acc_list)
    
    return 1.0 - mean_val_acc  # We want to maximize accuracy, so minimize 1 - accuracy

# Create the Optuna study and start optimization
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)  # You can adjust the number of trials

# Print the best hyperparameters
print("Best hyperparameters: ", study.best_params)

In [ ]:
layer_dims = [features.shape[1],8,class_num]
#print("Layer dimensions: ", layer_dims)
# the first layer will be big like 3703 neuron with 3703*F weight
base_model, acc_list, loss_list = train_base(
    layer_dims=layer_dims,
    data=(adj, features, labels, train_masks[0], val_masks[0] ),
    dropout_rate=0.017939,
    learning_rate=0.025421,
    weight_decay=3.306467e-05,
    self_importance=0.879005,
    epoch_num = 200
)

print("final accuracy:",acc_list[-1],"and final loss:",loss_list[-1])

In [ ]:
import matplotlib.pyplot as plt

# 绘制验证准确率曲线
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(acc_list, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy over Epochs')
plt.legend() #加图例

# 绘制损失值曲线
plt.subplot(1, 2, 2)
plt.plot(loss_list, label='Training Loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss over Epochs')
plt.legend()

plt.tight_layout() #调整子图布局
plt.show()